# TechSolve IT Support Intelligence Hub  
## Part 1 — Source, Combine & Prepare

This notebook follows a clear data-preparation workflow:

1. Load and inspect the raw Excel file.
2. Review rows, columns, data types, missing values and duplicate records.
3. Inspect category values, dates and numeric ranges.
4. Record initial data-quality findings.
5. Clean and standardise the ticket data.
6. Create consistent reporting categories and subcategories.
7. Add New Zealand public holiday information.
8. Add historical regional weather information.
9. Export cleaned datasets for the dashboard and AI agent.

## Optional Environment Setup — Install Required Python Packages

In [2]:
import sys
import subprocess
import importlib.util

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "holidays": "holidays",
    "requests": "requests",
}

for package_name, import_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_name}...")

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            package_name,
        ])
    else:
        print(f"{package_name} is already installed.")

print("\nPackage setup completed.")
print("Notebook environment:", sys.executable)

pandas is already installed.
numpy is already installed.
openpyxl is already installed.
holidays is already installed.
requests is already installed.

Package setup completed.
Notebook environment: /usr/local/bin/python3


## 1. Import libraries

In [3]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import holidays
import requests

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

## 2. Define file locations

This notebook assumes the following project structure:

```text
project-folder/
├── notebook/
│   └── data_preparation.ipynb
└── data/
    ├── raw/
    │   └── TechSolve - Ticket Data.xlsx
    ├── external/
    └── processed/
```

In [4]:
PROJECT_ROOT = Path.cwd()

# If this notebook is inside a notebooks folder, move one level up.
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "TechSolve - Ticket Data.xlsx"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw file:", RAW_FILE)

Project root: /Users/dhurandhar/Documents/GitHub/techsolve-it-support-intelligence-hub
Raw file: /Users/dhurandhar/Documents/GitHub/techsolve-it-support-intelligence-hub/data/raw/TechSolve - Ticket Data.xlsx


## 3. Load the raw Excel file

In [5]:
if not RAW_FILE.exists():
    raise FileNotFoundError(
        f"Raw Excel file not found: {RAW_FILE}\n"
        "Copy 'TechSolve - Ticket Data.xlsx' into data/raw/."
    )

raw_df = pd.read_excel(RAW_FILE, engine="openpyxl")

print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]:,}")

Rows: 100,851
Columns: 36


## 4. Inspect the first records and column names

In [6]:
raw_df.head()

,ticket_id,customer_id,customer_name,customer_email,company_name,account_type,customer_segment,industry,billing_contact_email,account_manager,monthly_contract_value,account_created_date,region,subscription_type,customer_tenure_months,service_area,category,priority,sla_target_hours,status,team,assigned_to,issue_description,resolution_notes,channel,ticket_created_date,ticket_resolved_date,first_response_time_hours,resolution_time_hours,escalated,sla_breached,csat_score,issue_complexity_score,previous_tickets,operating_system,browser
0,1,ACC-07512,David Martin,david.martin4@hotmail.com,Central Ltd,Business,Small Business,Retail,NaN,Sarah Chen,184.92,2018-11-21,Auckland,Premium,17,Cloud Storage,Security Concern,High,8,Closed,Support,Sam Mitchell,Two-factor authentication codes are not being ...,Data synchronization restored after backend se...,Social Media,2023-07-01,2023-07-07,67.66,19.11,No,No,3.0,6,12,Linux,Chrome
1,2,ACC-07455,Patricia Martin,patricia.martin869@icloud.com,Gateway Transport,Business,Small Business,Trades,NaN,NaN,0.00,2018-12-18,Wellington,Free,28,Cloud Storage,Performance Issue,Medium,24,Open,Billing,Nico Fernandez,I found a bug in the latest update affecting r...,Explained billing breakdown and clarified appl...,Phone,2023-07-01,2023-07-09,11.15,139.70,Yes,No,3.0,5,5,MacOS,Edge
2,3,ACC-03650,John Lopez,john.lopez551@gmail.com,Pioneer Logistics,Business,Small Business,Trades,NaN,NaN,0.00,2022-05-05,Auckland,Premium,54,Analytics Dashboard,Data Sync Issue,Low,48,Resolved,Support,Jordan Lee,The payment was deducted from my bank account ...,We have reset the account credentials and advi...,Chat,2023-07-01,2023-07-13,6.93,45.91,No,Yes,3.0,4,6,Linux,Safari
3,4,ACC-05579,Mary Martinez,mary.martinez515@company.com,Coastal Tech,Business,Small Business,Technology,NaN,NaN,0.00,2018-05-26,Southland,Free,59,Cloud Storage,Data Sync Issue,Medium,24,Open,Support,Drew Kapoor,My subscription was cancelled without my reque...,Subscription status corrected and confirmation...,Web Form,2023-07-01,2023-07-15,14.23,204.21,Yes,No,1.0,3,6,Linux,NaN
4,5,ACC-07456,John Martinez,john.martinez733@hotmail.com,NaN,Residential,Individual,NaN,NaN,NaN,49.28,2022-06-17,Wellington,Basic,15,API Service,Payment Problem,Urgent,4,Pending Customer,Billing,Priya Sharma,There seems to be a discrepancy in my billing ...,Explained billing breakdown and clarified appl...,Phone,2023-07-01,2023-07-09,68.76,96.30,No,No,1.0,8,10,Windows,Safari


In [7]:
raw_df.columns.tolist()

['ticket_id',
 'customer_id',
 'customer_name',
 'customer_email',
 'company_name',
 'account_type',
 'customer_segment',
 'industry',
 'billing_contact_email',
 'account_manager',
 'monthly_contract_value',
 'account_created_date',
 'region',
 'subscription_type',
 'customer_tenure_months',
 'service_area',
 'category',
 'priority',
 'sla_target_hours',
 'status',
 'team',
 'assigned_to',
 'issue_description',
 'resolution_notes',
 'channel',
 'ticket_created_date',
 'ticket_resolved_date',
 'first_response_time_hours',
 'resolution_time_hours',
 'escalated',
 'sla_breached',
 'csat_score',
 'issue_complexity_score',
 'previous_tickets',
 'operating_system',
 'browser']

In [8]:
raw_df.sample(5, random_state=42)

,ticket_id,customer_id,customer_name,customer_email,company_name,account_type,customer_segment,industry,billing_contact_email,account_manager,monthly_contract_value,account_created_date,region,subscription_type,customer_tenure_months,service_area,category,priority,sla_target_hours,status,team,assigned_to,issue_description,resolution_notes,channel,ticket_created_date,ticket_resolved_date,first_response_time_hours,resolution_time_hours,escalated,sla_breached,csat_score,issue_complexity_score,previous_tickets,operating_system,browser
59467,59468,ACC-06526,William Davis,william.davis372@gmail.com,NaN,Residential,Individual,NaN,NaN,NaN,0.00,2023-04-01,Canterbury,Basic,36,API Service,Subscription Cancellation,High,8,Resolved,NaN,NaN,I would like to request a refund for the recen...,We have reset the account credentials and advi...,Phone,2024-05-20,2024-05-25,66.26,162.83,Yes,Yes,4.0,7,4,Windows,Safari
12722,12723,ACC-07596,David Garcia,david.garcia571@hotmail.com,NaN,Residential,Individual,NaN,NaN,NaN,56.32,2023-11-10,Bay of Plenty,Free,42,E-commerce Store,Performance Issue,Low,48,Closed,NaN,NaN,The payment was deducted from my bank account ...,Provided step-by-step troubleshooting instruct...,Email,2023-09-08,2023-09-20,42.74,131.77,Yes,Yes,3.0,2,1,Linux,Firefox
19204,19205,ACC-06759,David Davis,david.davis117@company.com,Sterling Logistics,Business,Individual,Trades,NaN,Lisa Hartley,479.91,2023-01-19,Otago,Free,25,Payment Gateway,Login Issue,High,8,Pending Customer,NaN,NaN,The payment was deducted from my bank account ...,The duplicate charge was reversed and refund p...,Email,2023-10-14,2023-10-21,32.57,19.84,Yes,Yes,NaN,3,3,Android,NaN
1384,1385,ACC-08825,William Martinez,william.martinez198@yahoo.com,NaN,Residential,Individual,NaN,NaN,NaN,0.00,2022-02-25,Canterbury,Free,45,Mobile App,Subscription Cancellation,Low,48,Closed,NaN,NaN,I am unable to access my account after enterin...,Payment gateway timeout issue fixed and monito...,Email,2023-07-08,2023-07-17,2.27,179.15,Yes,Yes,3.0,3,20,Linux,Firefox
23433,23434,ACC-03687,Jennifer Hernandez,jennifer.hernandez609@outlook.com,Sterling Partners,Business,Corporate,Manufacturing,accounts@sterlingpartners.co.nz,NaN,75.69,2020-02-07,Auckland,Free,26,Subscription Service,Bug Report,Medium,24,In Progress,NaN,NaN,I would like to request a refund for the recen...,Explained billing breakdown and clarified appl...,Chat,2023-11-05,2023-11-07,51.85,191.41,Yes,Yes,4.0,9,2,Linux,Edge


## 5. Inspect data types and structure

In [9]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100851 entries, 0 to 100850
Data columns (total 36 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   ticket_id                  100851 non-null  int64         
 1   customer_id                100851 non-null  str           
 2   customer_name              100851 non-null  str           
 3   customer_email             100851 non-null  str           
 4   company_name               69856 non-null   str           
 5   account_type               100851 non-null  str           
 6   customer_segment           100851 non-null  str           
 7   industry                   69084 non-null   str           
 8   billing_contact_email      44488 non-null   str           
 9   account_manager            33895 non-null   str           
 10  monthly_contract_value     100851 non-null  float64       
 11  account_created_date       100851 non-null  datetime64[us]
 12 

In [10]:
data_types = (
    raw_df.dtypes
    .astype(str)
    .rename("data_type")
    .reset_index()
    .rename(columns={"index": "column"})
)

data_types

,column,data_type
0,ticket_id,int64
1,customer_id,str
2,customer_name,str
3,customer_email,str
4,company_name,str
5,account_type,str
6,customer_segment,str
7,industry,str
8,billing_contact_email,str
9,account_manager,str


## 6. Check missing values

In [11]:
missing_summary = pd.DataFrame({
    "missing_count": raw_df.isna().sum(),
    "missing_percentage": (raw_df.isna().mean() * 100).round(2),
})

missing_summary = (
    missing_summary
    .sort_values("missing_percentage", ascending=False)
    .reset_index()
    .rename(columns={"index": "column"})
)

missing_summary

,column,missing_count,missing_percentage
0,account_manager,66956,66.39
1,billing_contact_email,56363,55.89
2,industry,31767,31.50
3,company_name,30995,30.73
4,browser,21905,21.72
5,team,2017,2.00
6,assigned_to,2017,2.00
7,csat_score,2017,2.00
8,resolution_notes,1205,1.19
9,issue_complexity_score,0,0.00


## 7. Check duplicate records

In [12]:
print("Exact duplicate rows:", raw_df.duplicated().sum())

if "ticket_id" in raw_df.columns:
    print("Duplicate ticket IDs:", raw_df["ticket_id"].duplicated().sum())

Exact duplicate rows: 0
Duplicate ticket IDs: 0


## 8. Inspect important categorical columns

This step helps identify spelling variations and inconsistent labels before creating the category mapping.

In [13]:
categorical_columns = [
    "category",
    "service_area",
    "priority",
    "status",
    "team",
    "assigned_to",
    "channel",
    "region",
    "account_type",
    "customer_segment",
    "subscription_type",
    "industry",
    "escalated",
    "sla_breached",
    "operating_system",
    "browser",
]

for column in categorical_columns:
    if column in raw_df.columns:
        print(f"\n--- {column} ---")
        print(raw_df[column].value_counts(dropna=False).head(40))


--- category ---
category
Feature Request              8187
Payment Problem              8084
Performance Issue            8072
Account Suspension           8024
Subscription Cancellation    8009
Data Sync Issue              7997
Refund Request               7988
Security Concern             7975
Login Issue                  7857
Bug Report                   7826
Refund Req                   1074
Subscription Cancel          1065
account_suspension           1053
Perf Issue                   1037
feature_request              1036
sub_cancellation             1034
refund_request               1032
security_concern             1013
payment_problem              1013
Data Sync                    1007
Feature Req                  1005
Acct Suspension              1000
Security                      987
performance_issue             983
data_sync                     963
Payment problem               955
BUG                           788
LOGIN ISSUE                   785
Bug report           

## 9. Inspect date ranges

In [14]:
date_columns = [
    "account_created_date",
    "ticket_created_date",
    "ticket_resolved_date",
]

date_summary = []

for column in date_columns:
    if column in raw_df.columns:
        converted = pd.to_datetime(raw_df[column], errors="coerce")

        date_summary.append({
            "column": column,
            "minimum_date": converted.min(),
            "maximum_date": converted.max(),
            "invalid_or_missing": converted.isna().sum(),
        })

pd.DataFrame(date_summary)

,column,minimum_date,maximum_date,invalid_or_missing
0,account_created_date,2018-01-01,2023-12-31,0
1,ticket_created_date,1970-01-01,2099-12-31,0
2,ticket_resolved_date,1970-01-01,2025-03-18,0


In [15]:
ticket_created_preview = pd.to_datetime(
    raw_df["ticket_created_date"],
    errors="coerce"
)

ticket_created_preview.dt.year.value_counts(dropna=False).sort_index()

ticket_created_date
1970        5
2023    33676
2024    67157
2025       10
2099        3
Name: count, dtype: int64

## 10. Inspect numeric ranges

In [16]:
numeric_columns = [
    "monthly_contract_value",
    "customer_tenure_months",
    "sla_target_hours",
    "first_response_time_hours",
    "resolution_time_hours",
    "csat_score",
    "issue_complexity_score",
    "previous_tickets",
]

available_numeric_columns = [
    column for column in numeric_columns
    if column in raw_df.columns
]

raw_df[available_numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
monthly_contract_value,100851.0,202.818231,235.824310,0.0,0.00,109.46,298.670,986.98
customer_tenure_months,100851.0,30.314870,17.341046,1.0,15.00,30.00,45.000,60.00
sla_target_hours,100851.0,20.970184,17.283374,4.0,4.00,8.00,24.000,48.00
first_response_time_hours,100851.0,36.299138,20.649665,0.5,18.44,36.33,54.190,72.00
resolution_time_hours,100851.0,120.562948,68.893892,1.0,60.94,120.46,180.175,240.00
csat_score,98834.0,3.000840,1.414603,1.0,2.00,3.00,4.000,5.00
issue_complexity_score,100851.0,5.515553,2.873289,1.0,3.00,6.00,8.000,10.00
previous_tickets,100851.0,9.976460,6.061714,0.0,5.00,10.00,15.000,20.00


## 11. Initial findings

After running the inspection cells above, update this section with the exact findings from the data.

Typical issues to check include:

- duplicate rows or duplicate ticket IDs;
- inconsistent category spellings;
- missing team, analyst or CSAT values;
- dates outside the expected reporting period;
- invalid-looking dates such as 1970 or 2099;
- negative or unrealistic numeric values;
- resolved tickets without resolution dates;
- first response time greater than total resolution time;
- disagreement between the supplied SLA breach field and a recalculated SLA result.

The cleaning steps below retain questionable records and create flags rather than silently deleting them.

# Data cleaning

## 12. Create a working copy and clean column names

In [17]:
df = raw_df.copy()

df.columns = [
    re.sub(r"[^a-z0-9]+", "_", str(column).strip().lower()).strip("_")
    for column in df.columns
]

print(df.columns.tolist())

['ticket_id', 'customer_id', 'customer_name', 'customer_email', 'company_name', 'account_type', 'customer_segment', 'industry', 'billing_contact_email', 'account_manager', 'monthly_contract_value', 'account_created_date', 'region', 'subscription_type', 'customer_tenure_months', 'service_area', 'category', 'priority', 'sla_target_hours', 'status', 'team', 'assigned_to', 'issue_description', 'resolution_notes', 'channel', 'ticket_created_date', 'ticket_resolved_date', 'first_response_time_hours', 'resolution_time_hours', 'escalated', 'sla_breached', 'csat_score', 'issue_complexity_score', 'previous_tickets', 'operating_system', 'browser']


## 13. Remove exact duplicate rows

Duplicate ticket IDs are not automatically removed because they may require investigation.

In [18]:
original_row_count = len(df)
exact_duplicate_count = df.duplicated().sum()

df = df.drop_duplicates().copy()

print("Original rows:", f"{original_row_count:,}")
print("Exact duplicates removed:", f"{exact_duplicate_count:,}")
print("Rows after removal:", f"{len(df):,}")

Original rows: 100,851
Exact duplicates removed: 0
Rows after removal: 100,851


## 14. Clean text values

In [19]:
text_columns = df.select_dtypes(include=["object"]).columns

for column in text_columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

/var/folders/jd/1s0crjds2_s2kszkh09q3qx40000gn/T/ipykernel_78020/2088193131.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include=["object"]).columns


## 15. Convert dates and numeric columns

In [20]:
date_columns = [
    "account_created_date",
    "ticket_created_date",
    "ticket_resolved_date",
]

numeric_columns = [
    "monthly_contract_value",
    "customer_tenure_months",
    "sla_target_hours",
    "first_response_time_hours",
    "resolution_time_hours",
    "csat_score",
    "issue_complexity_score",
    "previous_tickets",
]

for column in date_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

## 16. Standardise controlled text fields

In [21]:
controlled_value_maps = {
    "priority": {
        "urgent": "Urgent",
        "high": "High",
        "medium": "Medium",
        "low": "Low",
    },
    "status": {
        "open": "Open",
        "in progress": "In Progress",
        "pending customer": "Pending Customer",
        "resolved": "Resolved",
        "closed": "Closed",
    },
    "escalated": {
        "yes": "Yes",
        "no": "No",
    },
    "sla_breached": {
        "yes": "Yes",
        "no": "No",
    },
}

for column, mapping in controlled_value_maps.items():
    if column in df.columns:
        lowercase_values = df[column].astype("string").str.lower()
        df[column] = lowercase_values.map(mapping).fillna(df[column])

## 17. Create consistent categories and subcategories

The mapping below is based on the category values observed during inspection.  
The original category remains available in `category_original`.

In [22]:
category_map = {
    "feature request": ("Product & Functionality", "Feature Request"),
    "feature req": ("Product & Functionality", "Feature Request"),
    "feature_request": ("Product & Functionality", "Feature Request"),

    "payment problem": ("Billing & Payments", "Payment Problem"),
    "payment_problem": ("Billing & Payments", "Payment Problem"),

    "performance issue": ("Performance & Availability", "Performance Issue"),
    "perf issue": ("Performance & Availability", "Performance Issue"),
    "performance_issue": ("Performance & Availability", "Performance Issue"),

    "account suspension": ("Account & Access", "Account Suspension"),
    "acct suspension": ("Account & Access", "Account Suspension"),
    "account_suspension": ("Account & Access", "Account Suspension"),

    "subscription cancellation": ("Subscription Management", "Cancellation"),
    "subscription cancel": ("Subscription Management", "Cancellation"),
    "sub_cancellation": ("Subscription Management", "Cancellation"),

    "data sync issue": ("Data & Integration", "Data Synchronisation"),
    "data sync": ("Data & Integration", "Data Synchronisation"),
    "data_sync": ("Data & Integration", "Data Synchronisation"),

    "refund request": ("Billing & Payments", "Refund Request"),
    "refund req": ("Billing & Payments", "Refund Request"),
    "refund_request": ("Billing & Payments", "Refund Request"),

    "security concern": ("Security & Privacy", "Security Concern"),
    "security": ("Security & Privacy", "Security Concern"),
    "security_concern": ("Security & Privacy", "Security Concern"),

    "login issue": ("Account & Access", "Login Issue"),
    "login_issue": ("Account & Access", "Login Issue"),

    "bug report": ("Product & Functionality", "Bug Report"),
    "bug": ("Product & Functionality", "Bug Report"),
    "bug_report": ("Product & Functionality", "Bug Report"),
}

In [23]:
df["category_original"] = df["category"]

category_key = (
    df["category"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["standard_category"] = category_key.map(
    lambda value: category_map.get(
        value,
        ("Other / Unclassified", "Manual Review")
    )[0]
)

df["standard_subcategory"] = category_key.map(
    lambda value: category_map.get(
        value,
        ("Other / Unclassified", "Manual Review")
    )[1]
)

df["category_review_required"] = (
    df["standard_category"] == "Other / Unclassified"
)

df[
    [
        "category_original",
        "standard_category",
        "standard_subcategory",
        "category_review_required",
    ]
].drop_duplicates().sort_values(
    ["standard_category", "standard_subcategory"]
)

,category_original,standard_category,standard_subcategory,category_review_required
12,account_suspension,Account & Access,Account Suspension,False
37,Account Suspension,Account & Access,Account Suspension,False
41,Acct Suspension,Account & Access,Account Suspension,False
19,Login Issue,Account & Access,Login Issue,False
29,Login issue,Account & Access,Login Issue,False
53,LOGIN ISSUE,Account & Access,Login Issue,False
163,login_issue,Account & Access,Login Issue,False
4,Payment Problem,Billing & Payments,Payment Problem,False
58,payment_problem,Billing & Payments,Payment Problem,False
185,Payment problem,Billing & Payments,Payment Problem,False


## 18. Add date-quality flags

The original dates are preserved. Clean date fields are created for reporting.

In [24]:
df["ticket_created_date_clean"] = df["ticket_created_date"]
df["ticket_resolved_date_clean"] = df["ticket_resolved_date"]

df["invalid_created_date"] = (
    df["ticket_created_date"].isna()
    | (df["ticket_created_date"].dt.year < 2020)
    | (df["ticket_created_date"].dt.year > 2026)
)

df["invalid_resolved_date"] = (
    df["ticket_resolved_date"].notna()
    & (
        (df["ticket_resolved_date"].dt.year < 2020)
        | (df["ticket_resolved_date"].dt.year > 2026)
    )
)

df.loc[
    df["invalid_created_date"],
    "ticket_created_date_clean"
] = pd.NaT

df.loc[
    df["invalid_resolved_date"],
    "ticket_resolved_date_clean"
] = pd.NaT

print("Invalid created dates:", df["invalid_created_date"].sum())
print("Invalid resolved dates:", df["invalid_resolved_date"].sum())

Invalid created dates: 8
Invalid resolved dates: 5


## 19. Add other data-quality checks

In [25]:
df["duplicate_ticket_id"] = df["ticket_id"].duplicated(keep=False)

df["missing_team"] = df["team"].isna()
df["missing_assigned_to"] = df["assigned_to"].isna()
df["missing_csat"] = df["csat_score"].isna()

df["resolved_before_created"] = (
    df["ticket_resolved_date_clean"].notna()
    & df["ticket_created_date_clean"].notna()
    & (
        df["ticket_resolved_date_clean"]
        < df["ticket_created_date_clean"]
    )
)

df["first_response_greater_than_resolution"] = (
    df["first_response_time_hours"].notna()
    & df["resolution_time_hours"].notna()
    & (
        df["first_response_time_hours"]
        > df["resolution_time_hours"]
    )
)

df["invalid_csat"] = (
    df["csat_score"].notna()
    & ~df["csat_score"].between(1, 5)
)

df["invalid_complexity_score"] = (
    df["issue_complexity_score"].notna()
    & ~df["issue_complexity_score"].between(1, 10)
)

df["negative_resolution_time"] = (
    df["resolution_time_hours"] < 0
)

df["negative_first_response_time"] = (
    df["first_response_time_hours"] < 0
)

## 20. Recalculate SLA breach for validation

The supplied field is retained. A separate calculated result is created so differences can be investigated.

In [26]:
df["calculated_sla_breach"] = np.where(
    df["resolution_time_hours"].notna()
    & df["sla_target_hours"].notna(),
    np.where(
        df["resolution_time_hours"] > df["sla_target_hours"],
        "Yes",
        "No",
    ),
    pd.NA,
)

df["sla_breach_mismatch"] = (
    df["calculated_sla_breach"].notna()
    & df["sla_breached"].notna()
    & (
        df["calculated_sla_breach"].astype("string")
        != df["sla_breached"].astype("string")
    )
)

print("SLA breach mismatches:", df["sla_breach_mismatch"].sum())

SLA breach mismatches: 50196


## 21. Create reporting columns

In [27]:
created = df["ticket_created_date_clean"]

df["ticket_date"] = created.dt.normalize()
df["ticket_year"] = created.dt.year.astype("Int64")
df["ticket_month_number"] = created.dt.month.astype("Int64")
df["ticket_month"] = created.dt.month_name()
df["ticket_year_month"] = created.dt.to_period("M").astype("string")
df["ticket_day_name"] = created.dt.day_name()
df["is_weekend"] = created.dt.dayofweek >= 5

df["is_reporting_period_2024_2025"] = (
    created.dt.year.between(2024, 2025)
)

df["is_open_ticket"] = df["status"].isin(
    ["Open", "In Progress", "Pending Customer"]
)

df["is_unassigned"] = (
    df["team"].isna()
    | df["assigned_to"].isna()
)

df["resolution_time_band"] = pd.cut(
    df["resolution_time_hours"],
    bins=[-np.inf, 2, 8, 24, 72, np.inf],
    labels=[
        "Under 2 hours",
        "2-8 hours",
        "8-24 hours",
        "1-3 days",
        "Over 3 days",
    ],
)

df["first_response_time_band"] = pd.cut(
    df["first_response_time_hours"],
    bins=[-np.inf, 1, 4, 8, 24, np.inf],
    labels=[
        "Under 1 hour",
        "1-4 hours",
        "4-8 hours",
        "8-24 hours",
        "Over 24 hours",
    ],
)

## 22. Review cleaned data

In [28]:
print(f"Cleaned rows: {len(df):,}")
print(
    "Rows in requested 2024-2025 period:",
    f"{df['is_reporting_period_2024_2025'].sum():,}"
)
print(
    "Categories requiring review:",
    f"{df['category_review_required'].sum():,}"
)

df.head()

Cleaned rows: 100,851
Rows in requested 2024-2025 period: 67,167
Categories requiring review: 0


,ticket_id,customer_id,customer_name,customer_email,company_name,account_type,customer_segment,industry,billing_contact_email,account_manager,monthly_contract_value,account_created_date,region,subscription_type,customer_tenure_months,service_area,category,priority,sla_target_hours,status,team,assigned_to,issue_description,resolution_notes,channel,ticket_created_date,ticket_resolved_date,first_response_time_hours,resolution_time_hours,escalated,sla_breached,csat_score,issue_complexity_score,previous_tickets,operating_system,browser,category_original,standard_category,standard_subcategory,category_review_required,ticket_created_date_clean,ticket_resolved_date_clean,invalid_created_date,invalid_resolved_date,duplicate_ticket_id,missing_team,missing_assigned_to,missing_csat,resolved_before_created,first_response_greater_than_resolution,invalid_csat,invalid_complexity_score,negative_resolution_time,negative_first_response_time,calculated_sla_breach,sla_breach_mismatch,ticket_date,ticket_year,ticket_month_number,ticket_month,ticket_year_month,ticket_day_name,is_weekend,is_reporting_period_2024_2025,is_open_ticket,is_unassigned,resolution_time_band,first_response_time_band
0,1,ACC-07512,David Martin,david.martin4@hotmail.com,Central Ltd,Business,Small Business,Retail,<NA>,Sarah Chen,184.92,2018-11-21,Auckland,Premium,17,Cloud Storage,Security Concern,High,8,Closed,Support,Sam Mitchell,Two-factor authentication codes are not being ...,Data synchronization restored after backend se...,Social Media,2023-07-01,2023-07-07,67.66,19.11,No,No,3.0,6,12,Linux,Chrome,Security Concern,Security & Privacy,Security Concern,False,2023-07-01,2023-07-07,False,False,False,False,False,False,False,True,False,False,False,False,Yes,True,2023-07-01,2023,7,July,2023-07,Saturday,True,False,False,False,8-24 hours,Over 24 hours
1,2,ACC-07455,Patricia Martin,patricia.martin869@icloud.com,Gateway Transport,Business,Small Business,Trades,<NA>,<NA>,0.00,2018-12-18,Wellington,Free,28,Cloud Storage,Performance Issue,Medium,24,Open,Billing,Nico Fernandez,I found a bug in the latest update affecting r...,Explained billing breakdown and clarified appl...,Phone,2023-07-01,2023-07-09,11.15,139.70,Yes,No,3.0,5,5,MacOS,Edge,Performance Issue,Performance & Availability,Performance Issue,False,2023-07-01,2023-07-09,False,False,False,False,False,False,False,False,False,False,False,False,Yes,True,2023-07-01,2023,7,July,2023-07,Saturday,True,False,True,False,Over 3 days,8-24 hours
2,3,ACC-03650,John Lopez,john.lopez551@gmail.com,Pioneer Logistics,Business,Small Business,Trades,<NA>,<NA>,0.00,2022-05-05,Auckland,Premium,54,Analytics Dashboard,Data Sync Issue,Low,48,Resolved,Support,Jordan Lee,The payment was deducted from my bank account ...,We have reset the account credentials and advi...,Chat,2023-07-01,2023-07-13,6.93,45.91,No,Yes,3.0,4,6,Linux,Safari,Data Sync Issue,Data & Integration,Data Synchronisation,False,2023-07-01,2023-07-13,False,False,False,False,False,False,False,False,False,False,False,False,No,True,2023-07-01,2023,7,July,2023-07,Saturday,True,False,False,False,1-3 days,4-8 hours
3,4,ACC-05579,Mary Martinez,mary.martinez515@company.com,Coastal Tech,Business,Small Business,Technology,<NA>,<NA>,0.00,2018-05-26,Southland,Free,59,Cloud Storage,Data Sync Issue,Medium,24,Open,Support,Drew Kapoor,My subscription was cancelled without my reque...,Subscription status corrected and confirmation...,Web Form,2023-07-01,2023-07-15,14.23,204.21,Yes,No,1.0,3,6,Linux,<NA>,Data Sync Issue,Data & Integration,Data Synchronisation,False,2023-07-01,2023-07-15,False,False,False,False,False,False,False,False,False,False,False,False,Yes,True,2023-07-01,2023,7,July,2023-07,Saturday,True,False,True,False,Over 3 days,8-24 hours
4,5,ACC-07456,John Martinez,john.martinez733@hotmail.com,<NA>,Residential,Individual,<NA>,<NA>,<NA>,49.28,2022-06-17,Wellington,Basic,15,API Service,Payment Problem,Urgent,4,Pending Customer,Billing,Priya Sharma,There seems to be a discrepancy in my billing ...,Expl

# External data enrichment

## 23. Add New Zealand public holidays

In [29]:
valid_years = (
    df["ticket_year"]
    .dropna()
    .astype(int)
    .sort_values()
    .unique()
    .tolist()
)

nz_holidays = holidays.NewZealand(
    years=valid_years,
    observed=True,
)

holiday_df = pd.DataFrame(
    [
        {
            "holiday_date": pd.Timestamp(day),
            "holiday_name": name,
        }
        for day, name in sorted(nz_holidays.items())
    ]
)

holiday_df["holiday_date"] = (
    pd.to_datetime(holiday_df["holiday_date"]).dt.normalize()
)

HOLIDAY_FILE = EXTERNAL_DIR / "nz_public_holidays.csv"
holiday_df.to_csv(HOLIDAY_FILE, index=False)

print("Holiday rows:", len(holiday_df))
print("Saved to:", HOLIDAY_FILE)

holiday_df.head()

Holiday rows: 34
Saved to: /Users/dhurandhar/Documents/GitHub/techsolve-it-support-intelligence-hub/data/external/nz_public_holidays.csv


,holiday_date,holiday_name
0,2023-01-01,New Year's Day
1,2023-01-02,Day after New Year's Day
2,2023-01-03,New Year's Day (observed)
3,2023-02-06,Waitangi Day
4,2023-04-07,Good Friday


In [30]:
df = df.merge(
    holiday_df,
    how="left",
    left_on="ticket_date",
    right_on="holiday_date",
)

df["is_public_holiday"] = df["holiday_name"].notna()

df["is_business_day"] = (
    df["ticket_date"].notna()
    & ~df["is_weekend"].fillna(False)
    & ~df["is_public_holiday"]
)

df[
    [
        "ticket_date",
        "holiday_name",
        "is_public_holiday",
        "is_business_day",
    ]
].head()

,ticket_date,holiday_name,is_public_holiday,is_business_day
0,2023-07-01,NaN,False,False
1,2023-07-01,NaN,False,False
2,2023-07-01,NaN,False,False
3,2023-07-01,NaN,False,False
4,2023-07-01,NaN,False,False


# Final validation and export

## 24. Create a final quality summary

In [35]:
quality_summary = pd.DataFrame({
    "check": [
        "Final rows",
        "Duplicate ticket IDs",
        "Invalid created dates",
        "Invalid resolved dates",
        "Missing team",
        "Missing assigned analyst",
        "Missing CSAT",
        "Category review required",
        "Resolved before created",
        "First response greater than resolution",
        "SLA breach mismatch",
        "Rows in 2024-2025",
        "Holiday matches",
    ],
    "count": [
        len(df),
        df["duplicate_ticket_id"].sum(),
        df["invalid_created_date"].sum(),
        df["invalid_resolved_date"].sum(),
        df["missing_team"].sum(),
        df["missing_assigned_to"].sum(),
        df["missing_csat"].sum(),
        df["category_review_required"].sum(),
        df["resolved_before_created"].sum(),
        df["first_response_greater_than_resolution"].sum(),
        df["sla_breach_mismatch"].sum(),
        df["is_reporting_period_2024_2025"].sum(),
        df["is_public_holiday"].sum(),
    ],
})

quality_summary

,check,count
0,Final rows,100851
1,Duplicate ticket IDs,0
2,Invalid created dates,8
3,Invalid resolved dates,5
4,Missing team,2017
5,Missing assigned analyst,2017
6,Missing CSAT,2017
7,Category review required,0
8,Resolved before created,50
9,First response greater than resolution,14750


## 25. Export the complete cleaned dataset

In [36]:
FULL_OUTPUT_FILE = (
    PROCESSED_DIR
    / "cleaned_enriched_ticket_data.csv"
)

df.to_csv(
    FULL_OUTPUT_FILE,
    index=False,
)

print("Saved full cleaned dataset to:")
print(FULL_OUTPUT_FILE)

Saved full cleaned dataset to:
/Users/dhurandhar/Documents/GitHub/techsolve-it-support-intelligence-hub/data/processed/cleaned_enriched_ticket_data.csv


## 26. Export the requested 2024–2025 reporting dataset

In [37]:
reporting_df = df.loc[
    df["is_reporting_period_2024_2025"].fillna(False)
].copy()

REPORTING_OUTPUT_FILE = (
    PROCESSED_DIR
    / "tickets_2024_2025.csv"
)

reporting_df.to_csv(
    REPORTING_OUTPUT_FILE,
    index=False,
)

print("Reporting rows:", f"{len(reporting_df):,}")
print("Saved reporting dataset to:")
print(REPORTING_OUTPUT_FILE)

Reporting rows: 67,167
Saved reporting dataset to:
/Users/dhurandhar/Documents/GitHub/techsolve-it-support-intelligence-hub/data/processed/tickets_2024_2025.csv


## 27. Export the quality summary

In [38]:
QUALITY_OUTPUT_FILE = (
    PROCESSED_DIR
    / "data_quality_summary.csv"
)

quality_summary.to_csv(
    QUALITY_OUTPUT_FILE,
    index=False,
)

print("Saved quality summary to:")
print(QUALITY_OUTPUT_FILE)

Saved quality summary to:
/Users/dhurandhar/Documents/GitHub/techsolve-it-support-intelligence-hub/data/processed/data_quality_summary.csv


# Part 1 outcome

At the end of this notebook, the project contains:

- the original untouched Excel file;
- a documented data-inspection process;
- cleaned ticket records;
- consistent reporting categories and subcategories;
- data-quality flags;
- New Zealand public holiday enrichment;
- historical regional weather enrichment;
- a complete cleaned dataset;
- a separate 2024–2025 reporting dataset;
- a data-quality summary.

The 2024–2025 dataset can be used for the dashboard, while the full enriched dataset can be retained for audit and further investigation.